# 02_bag_of_words_tfidf: Vector Representations using UCI SMS Spam Dataset
    
This notebook builds Bag of Words (BoW) and Term Frequency - Inverse Document Frequency (TF-IDF) representation matrices from scratch using NumPy over the real-world UCI SMS Spam dataset, comparing outputs against Scikit-Learn.


In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load UCI SMS Spam dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "message"])
print("Dataset size:", df.shape)

# Slice 5 sample messages to keep matrix prints readable
corpus_raw = df["message"].iloc[10:15].tolist()
# Basic normalization
corpus = [msg.lower().replace(".", "").replace(",", "") for msg in corpus_raw]

print("\nNormalized Corpus:")
for idx, doc in enumerate(corpus):
    print(f"Doc {idx+1}: {doc}")

# 2. Map vocabulary
import re
words = []
for doc in corpus:
    words.extend(re.findall(r"\b\w\w+\b", doc))
vocab = sorted(list(set(words)))
word_to_idx = {w: i for i, w in enumerate(vocab)}
print("\nVocabulary Mapping (Words -> Index):\n", word_to_idx)

# 3. Bag of Words (BoW) from scratch
bow_matrix = np.zeros((len(corpus), len(vocab)))
for doc_idx, doc in enumerate(corpus):
    for word in re.findall(r"\b\w\w+\b", doc):
        if word in word_to_idx:
            bow_matrix[doc_idx, word_to_idx[word]] += 1

print("\nBag of Words Matrix (from scratch):\n", bow_matrix)

# 4. Smooth TF-IDF from scratch
# Smooth IDF formulation: log((1 + N) / (1 + DF)) + 1
N = len(corpus)
df_counts = np.sum(bow_matrix > 0, axis=0)
idf = np.log((1 + N) / (1 + df_counts)) + 1

# Calculate TF-IDF
tfidf_matrix = bow_matrix * idf

# L2 normalization to match Scikit-Learn standard
norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
tfidf_norm = tfidf_matrix / (norms + 1e-15)  # prevent division by zero

print("\nTF-IDF Matrix (from scratch, normalized):\n", np.round(tfidf_norm, 4))

# 5. Compare with Scikit-Learn TfidfVectorizer
vectorizer = TfidfVectorizer(norm='l2', smooth_idf=True, use_idf=True)
sklearn_tfidf = vectorizer.fit_transform(corpus).toarray()
print("\nScikit-Learn TF-IDF Matrix:\n", np.round(sklearn_tfidf, 4))

# Check alignment
assert np.allclose(tfidf_norm, sklearn_tfidf, atol=1e-5)
print("\nSUCCESS: Custom TF-IDF matrix matches Scikit-Learn output exactly!")


Dataset size: (5572, 2)

Normalized Corpus:
Doc 1: i'm gonna be home soon and i don't want to talk about this stuff anymore tonight k? i've cried enough today
Doc 2: six chances to win cash! from 100 to 20000 pounds txt> csh11 and send to 87575 cost 150p/day 6days 16+ tsandcs apply reply hl 4 info
Doc 3: urgent! you have won a 1 week free membership in our £100000 prize jackpot! txt the word: claim to no: 81010 t&c wwwdbuknet lccltd pobox 4403ldnw1a7rw18
Doc 4: i've been searching for the right words to thank you for this breather i promise i wont take your help for granted and will fulfil my promise you have been wonderful and a blessing at all times
Doc 5: i have a date on sunday with will!!

Vocabulary Mapping (Words -> Index):
 {'100': 0, '100000': 1, '150p': 2, '16': 3, '20000': 4, '4403ldnw1a7rw18': 5, '6days': 6, '81010': 7, '87575': 8, 'about': 9, 'all': 10, 'and': 11, 'anymore': 12, 'apply': 13, 'at': 14, 'be': 15, 'been': 16, 'blessing': 17, 'breather': 18, 'cash': 19, 'chanc

### Output Explanation
- The custom matrix outputs align with `TfidfVectorizer` outputs exactly.
- Using a real SMS spam sample demonstrates how IDF weights down common words like `"to"` or `"you"` compared to unique message terms.
